## 과제 D: docstring 일부러 짧게 써보기 (실험 ★★★★★)

같은 도구의 docstring을 **자세한 버전 vs 짧은 버전**으로 만들어 비교.

```python
@tool
def add_v1(a: int, b: int) -> int:
    """더하기"""        # ← 너무 짧음
    return a + b

@tool
def add_v2(a: int, b: int) -> int:
    """두 정수의 합을 계산합니다.
    수학 계산이 필요한 모든 경우에 사용하세요."""
    return a + b
```

같은 질문에 LLM이 어떤 차이를 보이는지 관찰하세요.

---

> 💡 **하나만 골라서 깊이.** Pydantic 정형 반환 패턴이 손에 익도록.

In [1]:
# 1️⃣ 실행 환경 감지
try:
    import google.colab
    IS_COLAB = True
    print("✅ Google Colab")
except ImportError:
    IS_COLAB = False
    print("✅ 로컬 Jupyter")

# 3️⃣ API 키 로드
import os

if IS_COLAB:
    from google.colab import userdata
    os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
else:
    from dotenv import load_dotenv
    load_dotenv()

print("✅ API 키 OK" if os.environ.get("OPENAI_API_KEY") else "❌ 키 없음")

from typing import Literal, Optional
from pydantic import BaseModel, Field   # 1-2교시 정형화 도구
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage, AIMessage, ToolMessage
from langchain_core.tools import tool   # ⭐ 오늘의 주인공

# Tool Calling은 temperature=0 권장 (도구 선택의 일관성)
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

print("✅ import 완료")

✅ 로컬 Jupyter
✅ API 키 OK
✅ import 완료


In [9]:
def run_with_tools(question: str, tools: list, llm_obj=llm, max_steps: int = 5):
    """도구 호출이 끝날 때까지 반복 실행하고 최종 답변 반환"""
    llm_with_tools = llm_obj.bind_tools(tools)
    tool_map = {t.name: t for t in tools}
    messages = [HumanMessage(content=question)]

    for _ in range(max_steps):
        response = llm_with_tools.invoke(messages)
        messages.append(response)

        if not response.tool_calls:          
            return response.content

        for tc in response.tool_calls:
            print(f"  호출: {tc['name']} / 인자: {tc['args']}")
            result = tool_map[tc["name"]].invoke(tc["args"])

            error = getattr(result, "error", None)
            if error:
                return str(error)

            messages.append(
                ToolMessage(content=str(result), tool_call_id=tc["id"])
            )

    return "최대 단계 수를 초과했습니다."

print("✅ 헬퍼 함수 준비 완료")

✅ 헬퍼 함수 준비 완료


In [3]:
@tool
def add_v1(a: int, b: int) -> int:
    """더하기"""        # ← 너무 짧음
    return a + b

@tool
def add_v2(a: int, b: int) -> int:
    """두 정수의 합을 계산합니다.
    수학 계산이 필요한 모든 경우에 사용하세요."""
    return a + b

In [11]:
queries = [
    "1234와 5678을 더하면?",
    "사과가 17개 있고 25개를 더 받았어. 총 몇 개야?",
    "이번 달 교통비는 48000원, 식비는 137000원이야. 합계는?",
    "작년 방문자는 12500명, 올해는 17800명이야. 두 해의 총 방문자는?",
    "1234에서 567을 빼면?",
    "12와 8을 곱하면?",
]

for q in queries:
    print(f"\n질문: {q}")
    print("짧은 설명:", run_with_tools(q, [add_v1]))
    print("자세한 설명:", run_with_tools(q, [add_v2]))


질문: 1234와 5678을 더하면?
  호출: add_v1 / 인자: {'a': 1234, 'b': 5678}
짧은 설명: 1234와 5678을 더하면 6912입니다.
  호출: add_v2 / 인자: {'a': 1234, 'b': 5678}
자세한 설명: 1234와 5678을 더하면 6912입니다.

질문: 사과가 17개 있고 25개를 더 받았어. 총 몇 개야?
  호출: add_v1 / 인자: {'a': 17, 'b': 25}
짧은 설명: 총 사과는 42개입니다.
  호출: add_v2 / 인자: {'a': 17, 'b': 25}
자세한 설명: 사과는 총 42개입니다.

질문: 이번 달 교통비는 48000원, 식비는 137000원이야. 합계는?
  호출: add_v1 / 인자: {'a': 48000, 'b': 137000}
짧은 설명: 이번 달 교통비와 식비의 합계는 185,000원입니다.
  호출: add_v2 / 인자: {'a': 48000, 'b': 137000}
자세한 설명: 이번 달 교통비와 식비의 합계는 185,000원입니다.

질문: 작년 방문자는 12500명, 올해는 17800명이야. 두 해의 총 방문자는?
  호출: add_v1 / 인자: {'a': 12500, 'b': 17800}
짧은 설명: 작년과 올해의 총 방문자는 30,300명입니다.
  호출: add_v2 / 인자: {'a': 12500, 'b': 17800}
자세한 설명: 작년과 올해의 총 방문자는 30,300명입니다.

질문: 1234에서 567을 빼면?
  호출: add_v1 / 인자: {'a': 1234, 'b': -567}
짧은 설명: 1234에서 567을 빼면 667입니다.
  호출: add_v2 / 인자: {'a': 1234, 'b': -567}
자세한 설명: 1234에서 567을 빼면 667입니다.

질문: 12와 8을 곱하면?
  호출: add_v1 / 인자: {'a': 12, 'b': 8}
짧은 설명: 12와 8을 더하면 20입니다. 하지만 곱셈을 원하신다면,

In [12]:
for q in queries:
    print(f"\n질문: {q}")
    print("설명:", run_with_tools(q, [add_v1, add_v2]))


질문: 1234와 5678을 더하면?
  호출: add_v1 / 인자: {'a': 1234, 'b': 5678}
  호출: add_v2 / 인자: {'a': 1234, 'b': 5678}
설명: 1234와 5678을 더하면 6912입니다.

질문: 사과가 17개 있고 25개를 더 받았어. 총 몇 개야?
  호출: add_v1 / 인자: {'a': 17, 'b': 25}
설명: 사과는 총 42개입니다.

질문: 이번 달 교통비는 48000원, 식비는 137000원이야. 합계는?
  호출: add_v1 / 인자: {'a': 48000, 'b': 137000}
  호출: add_v2 / 인자: {'a': 48000, 'b': 137000}
설명: 이번 달 교통비와 식비의 합계는 185,000원입니다.

질문: 작년 방문자는 12500명, 올해는 17800명이야. 두 해의 총 방문자는?
  호출: add_v1 / 인자: {'a': 12500, 'b': 17800}
  호출: add_v2 / 인자: {'a': 12500, 'b': 17800}
설명: 작년과 올해의 총 방문자는 30,300명입니다.

질문: 1234에서 567을 빼면?
  호출: add_v1 / 인자: {'a': 1234, 'b': -567}
설명: 1234에서 567을 빼면 667입니다.

질문: 12와 8을 곱하면?
설명: 12와 8을 곱하면 96입니다.
